## Cell 1 — clone + install (idempotent)

Pulls the latest code from GitHub, installs dependencies into the Kaggle Python env, and registers the Atari ROMs. Safe to re-run.

In [ ]:
import os, shutil, subprocess, sys

REPO = 'spaceinvaderrl'
REPO_DIR = f'/kaggle/working/{REPO}'
CKPT_DIR = f'{REPO_DIR}/runs'

if os.path.isdir(REPO_DIR):
	import shutil as _s; _s.rmtree(REPO_DIR)
os.chdir('/kaggle/working')
!git clone https://github.com/FunctionalFalcon/{REPO}.git

os.chdir(REPO_DIR)
!git pull origin main || echo 'pull skipped (offline?)'

!pip install --no-deps --no-cache-dir -q -r requirements.txt 2>&1 | tail -5
!pip install --no-cache-dir -q gymnasium==0.29.1 ale-py==0.8.1 2>&1 | tail -5

import ale_py, gymnasium as gym
ale_py.register_v5_envs()
env = gym.make('ALE/SpaceInvaders-v5', frameskip=1, repeat_action_probability=0.25)
env.close()
print('envs registered OK')


## Cell 2 — smoketest (~3 min on T4)

Runs the from-scratch DQN for 5,000 steps to validate the whole pipeline before committing to the multi-hour run. Expected: a `.pt` saved under `runs/`, the eval CSV has 1 row, loss in 0.5-5.0 range, no NaNs.

Pipeline: env_fixed with min_repeat=3 → NatureCNN → Huber loss → Adam.

In [ ]:
os.chdir(REPO_DIR)
!python -m dqn.train --total_steps 5000 --save_freq 5000 --eval_freq 5000

import os
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_step_5000.pt'), 'no checkpoint'
assert os.path.exists(f'{CKPT_DIR}/dqn_scratch_eval.csv'), 'no eval csv'
print('\n[smoketest] OK')


## Cell 3 — full 2,000,000-step run on T4

Wall time estimate on T4: ~6-8 hours. Saves a `.pt` every 50k steps, runs 5 greedy eval episodes every 50k steps. Output dirs are `/kaggle/working/spaceinvaderrl_runs/` and `spaceinvaderrl_logs/`.

Trains against env_fixed (min_repeat=3) — the same wrapper that fixes the walk-off-screen bug from the SB3 baseline.

In [ ]:
os.chdir(REPO_DIR)
!python -m dqn.train --total_steps 2000000 --save_freq 50000 --eval_freq 50000


## Cell 4 — emergency resume (only if Cell 3 was interrupted)

If Kaggle disconnected you mid-run, the last completed checkpoint (`dqn_scratch_step_<N>.pt`) is what you resume from. The `--resume latest` flag auto-picks the most recent step checkpoint. Re-run Cell 4 until the total reaches 2,000,000.

In [ ]:
os.chdir(REPO_DIR)
!python -m dqn.train --resume latest --total_steps 2000000 --save_freq 50000 --eval_freq 50000


## Cell 5 — greedy evaluation (20 episodes)

Loads `runs/dqn_scratch_final.pt` and runs the from-scratch agent greedily for 20 episodes. Reports mean +/- std episode reward. Compare against the SB3 baseline (304.2 +/- 123.5) to see how close the from-scratch implementation gets.

`dqn/evaluate.py` reads `min_repeat` from the checkpoint's saved hp dict so eval env matches training env exactly.

In [ ]:
os.chdir(REPO_DIR)
!python -m dqn.evaluate --checkpoint {CKPT_DIR}/dqn_scratch_final.pt --episodes 20


## Cell 6 — record videos (one trained, one random baseline)

Records a 30fps MP4 of the trained agent + a random-policy baseline. Bypasses gymnasium's RecordVideo (fps=None crash on Kaggle) and uses imageio directly. The trained recorder uses env_fixed with min_repeat pulled from the checkpoint so the video matches what training saw.

In [ ]:
%%writefile /tmp/record_scratch_trained.py
"""Record a video of the FROM-SCRATCH DQN agent playing Space Invaders.

Loads a torch .pt checkpoint (not SB3's .zip), rebuilds QNetwork, and
writes an MP4 via imageio (gymnasium's RecordVideo crashes on Kaggle
because of an fps=None bug).

Uses env_fixed with min_repeat pulled from the checkpoint's saved hp
dict so the recorded env matches the training env exactly.
"""
from __future__ import annotations

import argparse, sys
from pathlib import Path

import imageio.v2 as imageio
import torch

from dqn.network import QNetwork
from dqn.hyperparam import Hyperparameters
from dqn.evaluate import greedy_action
from preprocessing import env_fixed

HERE = Path('/kaggle/working/spaceinvaderrl')
VIDEO_DIR = HERE / 'videos'

def main():
	p = argparse.ArgumentParser()
	p.add_argument('--checkpoint', type=str, required=True)
	p.add_argument('--episodes', type=int, default=1)
	p.add_argument('--name-prefix', type=str, default='trained-scratch')
	p.add_argument('--seed', type=int, default=42)
	p.add_argument('--fps', type=int, default=30)
	args = p.parse_args()

	VIDEO_DIR.mkdir(exist_ok=True)

	ckpt = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
	hp = Hyperparameters(**ckpt.get('hp', {}))
	num_actions = int(getattr(hp, 'num_actions', 6))
	min_repeat = int(getattr(hp, 'min_repeat', 4))
	q_net = QNetwork(num_actions=num_actions)
	q_net.load_state_dict(ckpt['model_state_dict'])
	q_net.eval()

	env = env_fixed(seed=args.seed, render_mode='rgb_array', min_repeat=min_repeat)
	for ep in range(args.episodes):
		out_path = VIDEO_DIR / f'{args.name_prefix}-episode-{ep}.mp4'
		print(f' Recording episode {ep+1} -> {out_path}')
		obs, _ = env.reset()
		total, steps, done = 0.0, 0, False
		with imageio.get_writer(out_path, fps=args.fps) as writer:
			while not done:
				writer.append_data(env.render())
				action = greedy_action(q_net, obs, device='cpu')
				obs, r, term, trunc, _ = env.step(action)
				total += float(r)
				steps += 1
				done = term or trunc
		print(f' Episode {ep+1}: {steps} steps, reward={total:.1f}')
		print(f' saved: {out_path}')
	env.close()
	return 0

if __name__ == '__main__':
	sys.exit(main())


In [ ]:
import os, subprocess, sys
os.chdir(REPO_DIR)

vdir = f'{REPO_DIR}/videos'
if os.path.isdir(vdir):
	for n in os.listdir(vdir):
		full = os.path.join(vdir, n)
		if os.path.isfile(full):
			os.remove(full)

subprocess.run([
	sys.executable, '/tmp/record_scratch_trained.py',
	'--checkpoint', f'{CKPT_DIR}/dqn_scratch_final.pt',
	'--episodes', '1',
])

# Random baseline: keep using make_env (no MinActionRepeat) so the
# comparison video shows raw random behavior, not random-with-commit.
!python {REPO_DIR}/record_random.py --episodes 1


## Cell 7 — zip everything for download

Kaggle's UI lets you download anything in `/kaggle/working`. This zips the videos + runs + logs into a single file you can grab at the end.

In [ ]:
import os
os.chdir('/kaggle/working')
out = '/kaggle/working/runs.zip'
if os.path.exists(out):
	os.remove(out)
!zip -r {out} {REPO}/videos {REPO}/runs spaceinvaderrl_runs spaceinvaderrl_logs
print(f'\n[done] {out} ({os.path.getsize(out):,} bytes)')
